In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#Copy datset from drive to local memory of t4 gpu
!mkdir -p "/content/dataset"
!cp -r -q "/content/drive/MyDrive/kisan_mitra_data/train" "/content/dataset/train"


In [ ]:
import os
import shutil
import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.cluster import KMeans

messy_dataset = '/content/dataset/train'
clean_dataset = '/content/dataset/Auto_Sorted_Data'
NUM_DISEASES = 10

feature_extractor = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))

image_paths = []
for root, dirs, files in os.walk(messy_dataset):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_paths.append(os.path.join(root, file))

BATCH_SIZE = 64
features = []
valid_paths = []

def load_and_prep(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    return preprocess_input(x)

for i in range(0, len(image_paths), BATCH_SIZE):
    batch_paths = image_paths[i:i + BATCH_SIZE]
    batch_images = []
    batch_valid_paths = []

    for p in batch_paths:
        try:
            batch_images.append(load_and_prep(p))
            batch_valid_paths.append(p)
        except Exception:
            continue

    if batch_images:
        batch_array = np.array(batch_images)
        # Predict the entire batch at once (lightning fast on T4 GPU)
        batch_features = feature_extractor.predict(batch_array, verbose=0)
        features.extend(batch_features)
        valid_paths.extend(batch_valid_paths)

features = np.array(features)



In [ ]:
kmeans = KMeans(n_clusters=NUM_DISEASES, random_state=42, n_init=10)
kmeans.fit(features)

print(f"K-Means clustering complete.")

In [ ]:
for i, cluster_label in enumerate(kmeans.labels_):
    cluster_dir = os.path.join(clean_dataset, f'Disease_Cluster_{cluster_label}')
    os.makedirs(cluster_dir, exist_ok=True)

    old_path = valid_paths[i]
    new_path = os.path.join(cluster_dir, os.path.basename(valid_paths[i]))
    
    try:
        shutil.copy(old_path, new_path)
    except Exception as e:
        print(f"Failed to copy {old_path}: {e}")
    
    if (i + 1) % 100 == 0:
        print(f"Organized {i + 1}/{len(valid_paths)} images")
